[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github.com/spirosChv/neuro208-tutorials/blob/main/Practical_NEURON_and_DendroTweaks/Channel_kinetics.ipynb)

In [71]:
import numpy as np
from bokeh.layouts import column, row
from bokeh.models import CustomJS, Slider, ColumnDataSource
from bokeh.plotting import figure, show, output_notebook
output_notebook()

In [72]:
# @markdown Initial data
v = np.linspace(-100, 100, 1000)
vhalf = 25
sigma = 10
k = 0.02
delta = 0.5
tau0 = 0

In [73]:
# @markdown Equations
xinf = 1 / (np.exp(-(v - vhalf) / sigma) + 1)

alpha_prime = k * np.exp(delta * (v - vhalf) / sigma)
beta_prime = k * np.exp(-(1 - delta) * (v - vhalf) / sigma)
taux = 1 / (alpha_prime + beta_prime) + tau0

In [74]:
# @markdown Bokeh elements

source = ColumnDataSource(data=dict(v=v, xinf=xinf, taux=taux))

p1 = figure(title="steady state", x_axis_label='V (mV)', y_axis_label='x_inf',
            height=400, width=400, y_range=(-0.1, 1.1))
p1.line('v', 'xinf', source=source, line_width=3, line_alpha=0.6)

p2 = figure(title="time constant", x_axis_label='V (mV)', y_axis_label='tau (ms)',
            height=400, width=400, y_range=(0, 100))
p2.line('v', 'taux', source=source, line_width=3, line_alpha=0.6)

vhalf_slider = Slider(start=-100, end=100, value=vhalf, step=1, title="V half")
sigma_slider = Slider(start=-50, end=50, value=sigma, step=1, title="sigma")
k_slider = Slider(start=0.005, end=1, value=k, step=0.001, title="K", format="0.000")
delta_slider = Slider(start=0, end=1, value=delta, step=0.01, title="delta")
tau0_slider = Slider(start=0, end=10, value=tau0, step=0.1, title="tau0")

callback = CustomJS(args=dict(source=source, vhalf=vhalf_slider, sigma=sigma_slider,
                             k=k_slider, delta=delta_slider, tau0=tau0_slider),
                    code="""
    const data = source.data;
    const v = data['v'];
    const xinf = data['xinf'];
    const taux = data['taux'];

    const Vh = vhalf.value;
    const s = sigma.value;
    const K = k.value;
    const d = delta.value;
    const t0 = tau0.value;

    for (let i = 0; i < v.length; i++) {
        // Update x_inf
        xinf[i] = 1 / (Math.exp(-(v[i] - Vh) / s) + 1);

        // Update alpha and beta primes
        let alpha = K * Math.exp(d * (v[i] - Vh) / s);
        let beta = K * Math.exp(-(1 - d) * (v[i] - Vh) / s);

        // Update tau
        taux[i] = 1 / (alpha + beta) + t0;
    }
    source.change.emit();
""")


for s in [vhalf_slider, sigma_slider, k_slider, delta_slider, tau0_slider]:
    s.js_on_change('value', callback)

layout = row(
    column(p1, vhalf_slider, sigma_slider),
    column(p2, k_slider, delta_slider, tau0_slider)
)

In [75]:
show(layout)